# 24 Gambar Reviewer


In [ ]:
# Jalur REVISED dan pembaca hasil praproses.
from pathlib import Path
import os, sys
from IPython.display import display
try:
    from google.colab import drive
except ImportError:
    pass
else:
    if not Path('/content/drive/MyDrive').is_dir():drive.mount('/content/drive',force_remount=False)
_explicit=os.environ.get('CARE_KM_ROOT') or os.environ.get('CARE_KM_PAPER_ROOT')
_options=([Path(_explicit)] if _explicit else []) + [Path('/content/drive/MyDrive/REVISED'),
    Path(__file__).resolve().parent.parent if '__file__' in globals() else Path.cwd(),Path.cwd(),Path.cwd().parent]
_ROOT_REVISED=next((p.resolve() for p in _options if all((p/d).is_dir() for d in ('DATA','LIB','PYTHON'))),None)
if _ROOT_REVISED is None:raise FileNotFoundError('Letakkan DATA, LIB, PYTHON di MyDrive/REVISED.')
sys.path.insert(0,str(_ROOT_REVISED/'LIB'))
# Jangan memakai modul versi lama yang masih tersimpan di memori Colab.
import importlib
_loaded=sys.modules.get('pipeline_paths')
if _loaded is None or (getattr(_loaded,'PIPELINE_LAYOUT_VERSION',None)!='2026.11.06-experiments-3'
    or Path(getattr(_loaded,'__file__','')).resolve()!=(_ROOT_REVISED/'LIB/pipeline_paths.py').resolve()):
    for _mod in ('pipeline_paths','pipeline_data','pipeline_runner','pipeline_cache','experiment_lib','care_km','generalization_data'):
        sys.modules.pop(_mod,None)
    importlib.invalidate_caches()
from pipeline_paths import configure,stage_dir,lms_data,pendikom_data,artifact_path,record_resample
configure(_ROOT_REVISED)


In [ ]:
from pipeline_data import preflight
preflight(_ROOT_REVISED,benchmarks=False)


In [ ]:
from pathlib import Path
import os, zipfile, shutil, glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# COLAB-READY SETUP
# ============================================================
# Notebook ini bisa dijalankan langsung di Google Colab.
# Jika folder OUTPUT belum ada, notebook akan:
# 1) mencari file ZIP CARE-KM di /content,
# 2) jika belum ada, meminta upload ZIP,
# 3) mengekstraknya,
# 4) menemukan folder OUTPUT secara otomatis.

OUTPUT = stage_dir('24_Gambar_Reviewer')
ROBUSTNESS_CSV = artifact_path('06_robustness_raw.csv')
RANK_CSV = artifact_path('13_table_07_generalization_average_rank.csv')
ROOT = OUTPUT.parent
FIG_DIR = artifact_path("FIGURES_REVIEWER",'24_Gambar_Reviewer')
FIG_DIR.mkdir(parents=True, exist_ok=True)

print("ROOT   :", ROOT)
print("OUTPUT :", OUTPUT)
print("FIG_DIR:", FIG_DIR)

# Verifikasi file sumber utama
required_files = [
    artifact_path("06_robustness_raw.csv",'24_Gambar_Reviewer'),
    artifact_path("13_table_07_generalization_average_rank.csv",'24_Gambar_Reviewer'),
]
missing = [str(p) for p in required_files if not p.exists()]
if missing:
    print("\nFile yang belum ditemukan:")
    for p in missing:
        print(" -", p)
    print("\nIsi OUTPUT yang tersedia:")
    for p in sorted(OUTPUT.iterdir()):
        print(" -", p.name)
    raise FileNotFoundError(
        "File sumber grafik tidak lengkap. Pastikan memakai ZIP OUTPUT CARE-KM yang sama dengan manuscript."
    )

print("\nSemua file sumber ditemukan. Lanjutkan ke cell berikutnya.")


rob = pd.read_csv(ROBUSTNESS_CSV)
required = {"rep", "method", "dbi", "si"}
missing = required.difference(rob.columns)
if missing:
    raise ValueError(f"Kolom robustness tidak lengkap: {sorted(missing)}")

method_order = ["CARE-KM", "Cao", "Bai", "K-Modes++", "Random", "Huang", "HR-2023"]
rob = rob[rob["method"].isin(method_order)].copy()
rob["resample"] = rob["rep"].astype(int) + 1

counts = rob.groupby("method")["resample"].nunique().reindex(method_order)
print("Jumlah resample per metode:")
print(counts.to_string())
if not (counts.dropna() == 30).all():
    print("PERINGATAN: tidak semua metode memiliki tepat 30 resample.")

# Figure 1a: DBI across 30 matched LMS resamples
fig, ax = plt.subplots(figsize=(10.8, 5.6))

for method in method_order:
    d = rob[rob["method"] == method].sort_values("resample")
    lw = 2.8 if method == "CARE-KM" else 1.25
    ms = 4.8 if method == "CARE-KM" else 3.2
    z = 5 if method == "CARE-KM" else 2
    ax.plot(d["resample"], d["dbi"], marker="o", markersize=ms,
            linewidth=lw, label=method, zorder=z)

ax.set_title("Matched-resample robustness: DBI across 30 LMS resamples")
ax.set_xlabel("Matched resample")
ax.set_ylabel("Davies-Bouldin index (lower is better)")
ax.set_xticks(range(1, 31, 2))
ax.grid(axis="y", alpha=0.25)
ax.legend(ncol=4, frameon=False, fontsize=9)
fig.tight_layout()

png = FIG_DIR / "reviewer_fig_01a_robustness_dbi_line.png"
pdf = FIG_DIR / "reviewer_fig_01a_robustness_dbi_line.pdf"
fig.savefig(png, dpi=600, bbox_inches="tight")
fig.savefig(pdf, bbox_inches="tight")
plt.show()
print("Saved:", png)
print("Saved:", pdf)

# Figure 1b: SI across 30 matched LMS resamples
fig, ax = plt.subplots(figsize=(10.8, 5.6))

for method in method_order:
    d = rob[rob["method"] == method].sort_values("resample")
    lw = 2.8 if method == "CARE-KM" else 1.25
    ms = 4.8 if method == "CARE-KM" else 3.2
    z = 5 if method == "CARE-KM" else 2
    ax.plot(d["resample"], d["si"], marker="o", markersize=ms,
            linewidth=lw, label=method, zorder=z)

ax.set_title("Matched-resample robustness: SI across 30 LMS resamples")
ax.set_xlabel("Matched resample")
ax.set_ylabel("Silhouette index (higher is better)")
ax.set_xticks(range(1, 31, 2))
ax.grid(axis="y", alpha=0.25)
ax.legend(ncol=4, frameon=False, fontsize=9)
fig.tight_layout()

png = FIG_DIR / "reviewer_fig_01b_robustness_si_line.png"
pdf = FIG_DIR / "reviewer_fig_01b_robustness_si_line.pdf"
fig.savefig(png, dpi=600, bbox_inches="tight")
fig.savefig(pdf, bbox_inches="tight")
plt.show()
print("Saved:", png)
print("Saved:", pdf)

rank_df = pd.read_csv(RANK_CSV)
required = {"source", "metric", "method", "average_rank"}
missing = required.difference(rank_df.columns)
if missing:
    raise ValueError(f"Kolom rank tidak lengkap: {sorted(missing)}")

metric_order = ["initial_cost", "final_cost", "dbi", "si", "iterations"]
metric_labels = ["Initial cost", "Final cost", "DBI", "SI", "Iterations"]

rank_plot = rank_df[
    (rank_df["source"] == "resample_mean") &
    (rank_df["metric"].isin(metric_order)) &
    (rank_df["method"].isin(method_order))
].copy()

pivot = rank_plot.pivot(index="metric", columns="method", values="average_rank").reindex(metric_order)
pivot = pivot.reindex(columns=method_order)
print(pivot.round(2).to_string())

if pivot.isna().any().any():
    raise ValueError("Ada nilai average rank yang hilang pada kombinasi metric-method.")

# Figure 2: grouped bar chart of average ranks
fig, ax = plt.subplots(figsize=(12.2, 6.2))

x = np.arange(len(metric_order))
width = 0.115
n_methods = len(method_order)

for j, method in enumerate(method_order):
    offset = (j - (n_methods - 1) / 2) * width
    bars = ax.bar(x + offset, pivot[method].values, width, label=method,
                  linewidth=1.4 if method == "CARE-KM" else 0.7,
                  hatch="///" if method == "CARE-KM" else None)
    if method == "CARE-KM":
        for bar, val in zip(bars, pivot[method].values):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.08,
                    f"{val:.2f}", ha="center", va="bottom", fontsize=9, fontweight="bold")

ax.set_title("Average ranks across ten categorical benchmark datasets")
ax.set_xlabel("Evaluation measure")
ax.set_ylabel("Average rank (lower is better)")
ax.set_xticks(x)
ax.set_xticklabels(metric_labels)
ax.set_ylim(0, 7.0)
ax.grid(axis="y", alpha=0.25)
ax.legend(ncol=4, frameon=False, fontsize=9)
fig.tight_layout()

png = FIG_DIR / "reviewer_fig_02_cross_dataset_average_rank_bar.png"
pdf = FIG_DIR / "reviewer_fig_02_cross_dataset_average_rank_bar.pdf"
fig.savefig(png, dpi=600, bbox_inches="tight")
fig.savefig(pdf, bbox_inches="tight")
plt.show()
print("Saved:", png)
print("Saved:", pdf)